In [1]:
import pandas as pd
from datetime import date, timedelta
import random
import numpy as np
np.random.seed(42)
random.seed(42)

Возьмем за основу "purchases_dataset.csv" - для имени, города, гендера

In [2]:
df = pd.read_csv("purchases_dataset.csv")
# удаляем данные с пропусками
df = df.dropna()

Создаем таблицу клиентов:
- client_id - идентификатор клиента
- full_name - полное имя
- birth_date - дата рождения
- gender - пол
- region - регион (город)
- registration_date - дата регистрации

In [3]:
# исходные данные
n = 1000 # количество записей
today = pd.Timestamp("2026-09-28") # фиксируем актуальную дату
mu, sigma = 40, 12 # некоторые параметры для нормального распределения возрастов


#  birth_date: возраст от 18 до 80, пик 40
#  registration_date: случайный возраст от 18 лет до сегодняшнего дня
birth_dates = pd.Series(dtype="datetime64[ns]")
registration_date = pd.Series(dtype="datetime64[ns]")
for k in range(n):
    p = 1
    while p: 
        age = random.gauss(mu, sigma)
        birth_dates.loc[k] = today - pd.Timedelta(days=round(age * 365))
        registration_date.loc[k] = birth_dates.loc[k] + pd.Timedelta(days=365*18+random.randint(1, (80-18)*365))
        if (age > 18) and (age < 80) and  (registration_date.loc[k] < today):
            p = 0



# собираем таблицу клиентов
client = pd.DataFrame({
    "client_id":         np.arange(1, n + 1),
    "full_name":         (df["customer_first_name"] + " " + df["customer_last_name"]).head(n).values,
    "birth_date":        birth_dates.values,
    "gender":            df["gender"].head(n).values,
    "region":            df["city"].head(n).values,
    "registration_date": registration_date.values
})

print(client.head())
print(client.dtypes)


   client_id          full_name birth_date gender       region  \
0          1  Barbara Hernandez 1976-01-16      F  San Antonio   
1          2  Matthew Rodriguez 1980-03-31      M     San Jose   
2          3  Matthew Rodriguez 1983-12-26      M     New York   
3          4     Patricia Davis 1970-04-14      F       Dallas   
4          5      Mary Anderson 1983-10-25      F      Chicago   

  registration_date  
0        1996-09-14  
1        2006-08-23  
2        2004-05-09  
3        2025-11-28  
4        2002-05-21  
client_id                     int64
full_name                    object
birth_date           datetime64[ns]
gender                       object
region                       object
registration_date    datetime64[ns]
dtype: object


Создаем список типов полисов - ОСАГО, КАСКО, ДМС, Страхование жизни, Имущество

In [4]:
policy_types = pd.DataFrame({
    "type_id": [1, 2, 3, 4, 5, 6], 
    "type_name": ["ОСАГО", "КАСКО", "ДМС", "Страхование жизни", "Несчастный случай", "Имущество"] 
})
print(policy_types)

   type_id          type_name
0        1              ОСАГО
1        2              КАСКО
2        3                ДМС
3        4  Страхование жизни
4        5  Несчастный случай
5        6          Имущество


Создаем список статусов полиса - активен, неактивен, удален

In [5]:
policy_statuses = pd.DataFrame({
    "status_id": [1, 2, 3],
    "status":    ["Активен", "Неактивен", "Удален"]
})

Создаем таблицу полисов:
- policy_id - идентификатор полиса
- client_id - идентификатор клиента
- policy_type - тип полиса (id из таблицы policy_types)
- start_date - начало действия полиса
- end_date - конец действия полиса
- premium - стоимость полиса
- status - статус полиса (id из таблицы policy_statuses)

In [6]:
n=2500
policy = pd.DataFrame({
    "policy_id" :np.arange(1, n + 1),
    "client_id": np.random.choice(client["client_id"],n),
    "policy_type": np.random.choice(policy_types["type_id"],n),
    "month": np.random.choice([12, 6, 24, 120, 1200], size=n, p=[0.90, 0.04, 0.04, 0.01, 0.01])
})
# month - длительность полиса в месяцах - 1 год, полгода, 2 года, 10 лет, 100 лет (=бессрочный)
reg = policy["client_id"].map(client.set_index("client_id")["registration_date"]) # даты регистрации клиентов для каждого полиса
days_range = (today - reg).dt.days.clip(lower=0) # возможные дни - от дня регистрации до сегодня
offsets = (np.random.rand(len(policy)) * days_range).astype(int) # список произвольных дней в указанном диапазоне
policy["start_date"] = (reg + pd.to_timedelta(offsets, unit="D")).values # дата полиса = дата регистрации + какое-то количество дней

policy["end_date"] = [
    sd + pd.DateOffset(months=int(m))
    for sd, m in zip(policy["start_date"], policy["month"])
]   
print(policy)                        

      policy_id  client_id  policy_type  month start_date   end_date
0             1        103            2     12 2008-04-03 2009-04-03
1             2        436            4     12 2026-03-15 2027-03-15
2             3        861            1     12 2007-08-06 2008-08-06
3             4        271            2     12 2023-08-29 2024-08-29
4             5        107            1      6 2024-09-03 2025-03-03
...         ...        ...          ...    ...        ...        ...
2495       2496        660            2     12 2013-10-19 2014-10-19
2496       2497        673            2     12 2011-06-26 2012-06-26
2497       2498        714            6     12 2005-07-08 2006-07-08
2498       2499        936            6     12 2026-02-12 2027-02-12
2499       2500        928            5     12 2018-01-17 2019-01-17

[2500 rows x 6 columns]


Стрим стоимости полисов, в зависимости от типа полиса

In [7]:
# осаго (1)
base_prices = np.random.choice(
    [14400, 10200, 8700, 7000], # цены 
    size=n,
    p=[0.2, 0.2, 0.2, 0.4]      # вероятности 
)
multiplier = np.random.uniform(0.85, 1.15, size=n) # Элемент случайности
premiums_osago = (base_prices * multiplier).round(2)


In [8]:
# каско (2)
car_prices = np.random.uniform(400_000, 5_000_000, size=n)
rate = np.random.uniform(0.03, 0.12, size=n)
premium_kasko = (car_prices * rate * np.random.uniform(0.95, 1.05, size=n)).round(2)


In [9]:
# дмс (3)
program_types = np.random.choice(
    ["budget", "standard", "comfort", "premium"], 
    size=n, 
    p=[0.2, 0.5, 0.2, 0.1] # 20% - базовые, 50% - стандарт, 20% - комфорт, 10% - премиум
)

# диапазоны цен для каждого типа
price_ranges = {
    "budget":   (3000, 15000),    
    "standard": (15000, 45000),   
    "comfort":  (45000, 100000), 
    "premium":  (100000, 200000), 
}

# цены для каждой строки
premiums = []
for p_type in program_types:
    low, high = price_ranges[p_type]
    price = np.random.triangular(low, (low + high) / 2, high)
    premiums.append(price)

premium_dms = np.array(premiums).round(2)


In [10]:
# страхование жизни (4)
policy_sl_types = np.random.choice([0, 1, 2], size=n, p=[0.7, 0.25, 0.05])

# диапазоны цен для каждого типа 
price_ranges = {
    0: (1400, 12000),    
    1: (1400, 4500),       
    2: (50000, 500000),    
}

# генерируем цену
premiums = []
for p_type in policy_sl_types:
    low, high = price_ranges[p_type]
    price = np.random.triangular(low, (low + high) / 2, high)
    premiums.append(price)

life_premium = np.array(premiums).round(2)


In [11]:
# несчастный случай (5)

premiums = np.random.normal(loc=2000, scale=400, size=n)
premiums_ns = np.clip(premiums, 1500, 5000).round(2)


In [12]:
# имущество (6)
premiums = np.random.lognormal(mean=9.2, sigma=0.8, size=n)
premiums_h = np.clip(premiums, 2000, 70000).round(2)


In [13]:

policy["premium"] = np.select(
    [
        policy["policy_type"] == 1,   # ОСАГО
        policy["policy_type"] == 2,   # КАСКО
        policy["policy_type"] == 3,   # ДМС
        policy["policy_type"] == 4,   # жизнь
        policy["policy_type"] == 5,   # НС
        policy["policy_type"] == 6,   # имущество
    ],
    [
       premiums_osago,      # массив/Series той же длины
       premium_kasko,
       premium_dms,
       life_premium,
       premiums_ns,
       premiums_h,
    ],
    default=np.nan          # если ни одно условие не сработало
)

# зависимость от длительности действия полиса
policy["premium"] = policy["premium"]*policy["month"]/12
policy.loc[policy["month"] == 1200, "premium"]=round(policy.loc[policy["month"] == 1200, "premium"]/5, 2)


Проставляем статусы полисов
- у кого действие не закончилось - активыне
- у кого действие закончилось - неактивные
- 5% от всех - удаленные

In [14]:
policy["status"] = 1
policy.loc[policy["end_date"] < today, "status"] = 2
policy.loc[np.random.rand(len(policy)) < 0.05, "status"] = 3

In [15]:
policy  = policy.drop(columns =["month"])

Создаем таблицу payments:
- payment_id - идентификатор платежа
- policy_id - полис по которому произведен платеж
- payment_date - дата платежа
- amount - сумма платежа
- status - статус платежа

In [16]:
n = 7500
payment1 = pd.DataFrame({
    "payment_id" :np.arange(1, n + 1),
    "policy_id": np.random.choice(policy["policy_id"],n)
})
print(payment1)


      payment_id  policy_id
0              1       1748
1              2       2309
2              3       1838
3              4       1024
4              5       1641
...          ...        ...
7495        7496       2347
7496        7497       2140
7497        7498        750
7498        7499       1300
7499        7500        304

[7500 rows x 2 columns]


Дата платежа - между регистрацией и началом действия полиса

In [17]:
reg = (
    payment1[["policy_id"]]
    .merge(policy[["policy_id", "client_id", "start_date"]], on="policy_id", how="left")
    .merge(client[["client_id", "registration_date"]], on="client_id", how="left")
)

lo = reg["registration_date"]
hi = reg["start_date"]
days_range = (hi - lo).dt.days.clip(lower=0)
offsets = (np.random.rand(len(reg)) * days_range).astype(int)

payment1["payment_date"] = (lo + pd.to_timedelta(offsets, unit="D")).values

In [18]:
tmp = payment1["policy_id"].value_counts().reset_index()
#print(tmp[1802])
for t in tmp:
    print(t)

policy_id
count


In [19]:
tmp1 = payment1.copy()

In [20]:
tmp1["t1"] = np.random.rand(len(tmp1))
tmp1 = tmp1.merge(
    policy[["policy_id", "premium"]],
    on = "policy_id"
).merge(
    tmp[["policy_id", "count"]],
    on = "policy_id"
)
print(tmp1)

      payment_id  policy_id payment_date        t1    premium  count
0              1       1748   2017-04-01  0.556548    1781.75      1
1              2       2309   2019-07-05  0.723338  403223.02      3
2              3       1838   2026-08-25  0.416746   14544.76      3
3              4       1024   2019-06-26  0.325849    6599.11      5
4              5       1641   2014-12-15  0.916577    2000.00      7
...          ...        ...          ...       ...        ...    ...
7495        7496       2347   2012-11-15  0.503558    6503.30      3
7496        7497       2140   2019-12-03  0.358649   27678.88      6
7497        7498        750   2008-09-18  0.156112    3562.51      3
7498        7499       1300   2012-08-31  0.367131   12763.07      5
7499        7500        304   2026-05-01  0.609104    7013.93      4

[7500 rows x 6 columns]


In [21]:
tmp3 = tmp1.groupby("policy_id")["t1"].sum().reset_index()
tmp3 = tmp3.rename(columns={"t1":"t1_sum"})
tmp1 = tmp1.merge(
    tmp3,
    on = "policy_id"
)



In [22]:
tmp1["prem"] = round(tmp1["premium"]*tmp1["t1"]/tmp1["t1_sum"],2)
print(tmp1)

      payment_id  policy_id payment_date        t1    premium  count  \
0              1       1748   2017-04-01  0.556548    1781.75      1   
1              2       2309   2019-07-05  0.723338  403223.02      3   
2              3       1838   2026-08-25  0.416746   14544.76      3   
3              4       1024   2019-06-26  0.325849    6599.11      5   
4              5       1641   2014-12-15  0.916577    2000.00      7   
...          ...        ...          ...       ...        ...    ...   
7495        7496       2347   2012-11-15  0.503558    6503.30      3   
7496        7497       2140   2019-12-03  0.358649   27678.88      6   
7497        7498        750   2008-09-18  0.156112    3562.51      3   
7498        7499       1300   2012-08-31  0.367131   12763.07      5   
7499        7500        304   2026-05-01  0.609104    7013.93      4   

        t1_sum       prem  
0     0.556548    1781.75  
1     1.843665  158199.24  
2     2.134628    2839.59  
3     2.211889     972.

In [23]:
sums = tmp1.groupby("policy_id")["prem"].sum()
policy["premium1"] = policy["policy_id"].map(sums).fillna(0)
policy["premium"]=np.where(
    policy["premium1"]>0,
    policy["premium1"],
    policy["premium"]
)
policy["premium"] = policy["premium"].round(2)

mask = policy["premium1"] == 0
n = mask.sum()

policy.loc[mask, "status"] = np.random.choice([2, 3], size=n)
policy = policy.drop(columns = ["premium1"])
print(policy)

      policy_id  client_id  policy_type start_date   end_date    premium  \
0             1        103            2 2008-04-03 2009-04-03  132422.48   
1             2        436            4 2026-03-15 2027-03-15    3652.24   
2             3        861            1 2007-08-06 2008-08-06   13654.26   
3             4        271            2 2023-08-29 2024-08-29  267122.32   
4             5        107            1 2024-09-03 2025-03-03    5620.88   
...         ...        ...          ...        ...        ...        ...   
2495       2496        660            2 2013-10-19 2014-10-19   71492.66   
2496       2497        673            2 2011-06-26 2012-06-26  240852.19   
2497       2498        714            6 2005-07-08 2006-07-08    3459.78   
2498       2499        936            6 2026-02-12 2027-02-12   10569.12   
2499       2500        928            5 2018-01-17 2019-01-17    2349.88   

      status  
0          2  
1          1  
2          2  
3          2  
4          2

In [24]:
n4 = 500
tmp4 = pd.DataFrame({
    "policy_id": np.random.choice(policy["policy_id"],n4)
})

tmp4 = tmp4.merge(
    policy[["policy_id", "premium", "start_date", "client_id"]],
    on = "policy_id"
).merge(
    client[["client_id", "registration_date"]],
    on = "client_id"
)
print(tmp4)


     policy_id    premium start_date  client_id registration_date
0         1460    9820.91 2024-12-01        821        2018-11-28
1         2302    2264.78 2011-11-09        831        2011-10-11
2         1789    2240.34 2024-03-22        506        2014-06-25
3          776    4000.01 2018-07-16        273        2015-11-25
4          316    1500.00 2024-03-09        198        2019-11-12
..         ...        ...        ...        ...               ...
495       1021    2366.88 2001-01-25        929        2000-07-08
496        574   42375.85 2012-04-13        993        2011-02-27
497       1943    6055.45 2022-07-06        957        1998-08-19
498       2397  259695.80 2026-06-28        257        2022-07-17
499       2064    1831.39 2018-09-05        748        2013-03-22

[500 rows x 5 columns]


In [25]:

days = (tmp4["start_date"]- tmp4["registration_date"]).dt.days.clip(lower=0)
offset = (np.random.rand(len(tmp4))*days).astype(int)
tmp4["payment_date"] = (tmp4["registration_date"]+pd.to_timedelta(offset, unit="D")).values
tmp4["amount"] = (tmp4["premium"] * np.random.rand(len(tmp4))).round(2)

tmp4["status"] = np.random.choice(
    [2, 3],
    size=len(tmp4),
    p=[0.34, 0.66]  
)
tmp1 = tmp1.rename(
    columns = {"prem":"amount"}
)
tmp1["status"]=1
tmp5 = pd.concat([tmp1[["policy_id", "payment_date","amount", "status"]], 
                  tmp4[["policy_id", "payment_date", "amount", "status"]]],axis=0)
tmp5 = tmp5.sample(frac=1).reset_index(drop=True)


In [26]:
payment = tmp5.copy()   # чтобы не менять tmp5
payment.insert(0, "payment_id", np.arange(1, len(payment) + 1))


Создаем таблицу выплат по ображениям по страховым случаям claim:
- claim_id - идентификатор выплаты
- policy_id - полис по которому производится выплата
- claim_date - дата выплаты
- claim_type - тип выплаты (из  таблицы claim_types)
- claim_amount - сумма выплаты
- status - статус выплаты (из таблицы claim_statuses)

Типы выплат по соответствующему полису\
1 - 1 - (осаго) дтп\
2 - 1 - (осаго) вред здоровью\
3 - 1 - (осаго) повреждение имущества\
4 - 2 - (каско) дтп\
5 - 2 - (каско) угон\
6 - 2 - (каско) повреждение автомобиля\
7 - 2 - (каско) пожар\
8 - 2 - (каско) стихийное бедствие\
9 - 3 - (дмс) амбулаторное лечение\
10 - 3 - (дмс) госпитализация\
11 - 3 - (дмс) диагностика\
12 - 3 - (дмс) стоматология\
13 - 4 - (страхование жизни) смерть\
14 - 4 - (страхование жизни) критическое заболевание\
15 - 4 - (страхование жизни) дожитие\
16 - 5 - (несчастный случай) травма\
17 - 5 - (несчастный случай) перелом\
18 - 5 - (несчастный случай) инвалидность\
19 - 6 - (страхование имущества) пожар\
20 - 6 - (страхование имущества) затопление\
21 - 6 - (страхование имущества) кража\
22 - 6 - (страхование имущества) повреждение имущества\
23 - 6 - (страхование имущества) стихийное бедствие\

In [27]:
n_cl = 750
cl1 = pd.DataFrame({
    "claim_id" :np.arange(1, n_cl + 1),
    "policy_id": np.random.choice(policy["policy_id"],n_cl)
})
cl1 = cl1.merge(
    policy[["policy_id", "policy_type", "start_date", "end_date", "premium"]],
    on = "policy_id"
)
print(cl1)

     claim_id  policy_id  policy_type start_date   end_date    premium
0           1       2195            3 2016-08-28 2017-08-28   34458.58
1           2       2209            5 2025-12-12 2026-12-12    2130.29
2           3        718            2 2020-02-09 2021-02-09  382391.82
3           4       1293            6 2025-12-03 2026-12-03    6756.46
4           5       1287            1 2015-10-07 2016-10-07    8043.13
..        ...        ...          ...        ...        ...        ...
745       746       1809            5 2025-06-27 2026-06-27    2109.14
746       747        421            1 2016-05-04 2017-05-04    8625.27
747       748       1964            1 2026-01-22 2027-01-22   14210.39
748       749       1047            5 2026-06-16 2027-06-16    1837.09
749       750       2344            4 2021-09-07 2023-09-07    5133.12

[750 rows x 6 columns]


In [28]:

# верхняя граница для каждой строки: min(end_date, today)
hi = cl1[["end_date"]].assign(today=today).min(axis=1)

# диапазон в днях
lo = cl1["start_date"]
days_range = (hi - lo).dt.days

# случайный сдвиг
offsets = (np.random.rand(len(cl1)) * days_range).astype(int)

# claim_date
cl1["claim_date"] = lo + pd.to_timedelta(offsets, unit="D")

In [29]:
claim_type_map = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8],
    3: [9, 10, 11, 12],
    4: [13, 14, 15],
    5: [16, 17, 18],
    6: [19, 20, 21, 22, 23],
}

cl1["claim_type"] = [
    random.choice(claim_type_map[pt]) if pt in claim_type_map else np.nan
    for pt in cl1["policy_type"]
]


In [30]:
# минимальный и максмальный коэффициент выплаты от премии полиса за год
data = [
    (1, 0.5, 5.0),
    (2, 0.3, 3.0),
    (3, 0.2, 2.5),
    (4, 0.5, 5.0),
    (5, 2.0, 8.0),
    (6, 0.2, 3.0),
    (7, 1.0, 6.0),
    (8, 0.5, 5.0),
    (9, 0.01, 0.3),
    (10, 0.1, 1.0),
    (11, 0.1, 0.2),
    (12, 0.02, 0.4),
    (13, 5.0, 20.0),
    (14, 2.0, 10.0),
    (15, 1.0, 5.0),
    (16, 0.1, 2.0),
    (17, 0.2, 3.0),
    (18, 1.0, 8.0),
    (19, 1.0, 6.0),
    (20, 0.2, 4.0),
    (21, 0.2, 3.0),
    (22, 0.2, 2.5),
    (23, 0.5, 5.0)
]

# как таблица
df = pd.DataFrame(data, columns=["claim_type", "min", "max"])


In [31]:
cl1 = cl1.merge(
    df,
    on="claim_type"
)


In [32]:
cl1["k"] = np.random.uniform(cl1["min"], cl1["max"])
cl1["years"] = ((cl1["end_date"] - cl1["start_date"]).dt.days / 365).round(3)
cl1["claim_amount"] = (cl1["k"]*cl1["premium"]/cl1["years"]).round(2)


In [ ]:
cl1["status"] = np.random.choice(
    [1, 2, 3],
    size=len(cl1),
    p=[0.8, 0.15, 0.05]
)

In [ ]:
allow_days = today - pd.Timedelta(days=120)
cl1.loc[cl1["claim_date"] < allow_days, "status"] = 1


In [ ]:
claim = cl1[["claim_id", "policy_id", "claim_date", "claim_type", "claim_amount", "status"]]

Сохраняем все полученные таблицы

In [824]:
print(policy)

      policy_id  client_id  policy_type start_date   end_date   premium  \
0             1        277            6 2023-08-21 2024-08-21   3009.16   
1             2        337            2 2024-11-05 2025-11-05  63806.30   
2             3        620            3 2023-10-02 2024-10-02   9558.08   
3             4        704            3 2024-08-28 2025-08-28  11802.24   
4             5        548            3 2016-10-26 2017-10-26  31031.62   
...         ...        ...          ...        ...        ...       ...   
2495       2496        368            1 2026-01-21 2027-01-21  16474.14   
2496       2497        170            5 2021-02-09 2022-02-09   2087.05   
2497       2498         90            1 2025-10-08 2026-10-08   7538.60   
2498       2499        799            4 2026-02-01 2027-02-01   8956.26   
2499       2500        501            6 2025-07-28 2026-07-28  20843.00   

      status  
0          2  
1          2  
2          2  
3          2  
4          2  
...      

In [825]:
policy.to_csv("policy.csv", index = False)

In [826]:
print(client)

     client_id          full_name birth_date gender       region  \
0            1         Mary Davis 1975-03-24      F    San Diego   
1            2  Barbara Hernandez 1988-08-06      F  San Antonio   
2            3  Matthew Rodriguez 1982-01-10      M     San Jose   
3            4  Matthew Rodriguez 1996-07-17      M     New York   
4            5     Patricia Davis 1986-03-09      F       Dallas   
..         ...                ...        ...    ...          ...   
995        996   Charles Williams 1978-08-16      M    San Diego   
996        997  Patricia Williams 1988-03-17      F      Houston   
997        998    David Hernandez 1985-12-17      M      Houston   
998        999  Barbara Hernandez 1993-04-23      F     New York   
999       1000   Elizabeth Miller 1978-10-26      F      Houston   

    registration_date  
0          2006-01-16  
1          2021-06-09  
2          2019-03-06  
3          2014-11-13  
4          2025-09-23  
..                ...  
995        2012

In [827]:
client.to_csv("client.csv", index = False)

In [828]:
print(payment)

      payment_id  policy_id payment_date    amount  status
0              1       1319   2022-08-08   4988.49       3
1              2       2317   2019-11-21  32593.80       1
2              3       2184   2022-08-21    578.18       1
3              4        381   2013-03-13   1749.24       1
4              5       2319   2004-08-28  63484.49       1
...          ...        ...          ...       ...     ...
7995        7996        390   2006-10-09    900.20       1
7996        7997       1902   2010-11-08  12764.69       1
7997        7998       2427   2022-08-13   3891.55       3
7998        7999       2287   2016-12-12   1718.86       1
7999        8000        137   1991-08-04   2450.95       1

[8000 rows x 5 columns]


In [829]:
payment.to_csv("payment.csv", index = False)

In [830]:
print(claim)

     claim_id  policy_id claim_date  claim_type  claim_amount  status
0           1       1436 2022-10-09          22       5357.70       1
1           2        700 2021-04-16          11       3323.89       1
2           3       1532 2016-02-08          15      12636.78       1
3           4        434 2019-11-18          21      11189.61       1
4           5       1374 2025-02-23          13      63503.49       1
..        ...        ...        ...         ...           ...     ...
745       746       2246 2024-04-21          20      72966.30       1
746       747       2053 2026-06-16           8     574743.15       2
747       748       1550 2026-01-14           3      11328.72       1
748       749        690 2024-03-16          14      48710.62       1
749       750        545 2009-06-09           7     849128.07       1

[750 rows x 6 columns]


In [831]:
claim.to_csv("claim.csv", index = False)

In [837]:
policy_types_list = [
    (1, "ОСАГО"),
    (2, "КАСКО"),
    (3, "ДМС"),
    (4, "Страхование жизни"),
    (5, "Несчастный случай"),
    (6, "Страхование имущества")
]
# как таблица
policy_types = pd.DataFrame(policy_types_list, columns=["type_id", "type_name"])
policy_types.to_csv("policy_types.csv", index = False)

In [839]:
policy_statuses_list = [
    (1, "Активен"),
    (2, "Неактивен"),
    (3, "Удален")
]

policy_statuses = pd.DataFrame(policy_statuses_list, columns=["status_id", "status"])
policy_statuses.to_csv("policy_statuses.csv", index = False)

In [840]:
payment_statuses_list = [
    (1, "Успешен"),
    (2, "Неуспешен"),
    (3, "Удален")
]

payment_statuses = pd.DataFrame(payment_statuses_list, columns=["status_id", "status"])
payment_statuses.to_csv("payment_statuses.csv", index = False)

1 - 1 - (осаго) дтп\
2 - 1 - (осаго) вред здоровью\
3 - 1 - (осаго) повреждение имущества\
4 - 2 - (каско) дтп\
5 - 2 - (каско) угон\
6 - 2 - (каско) повреждение автомобиля\
7 - 2 - (каско) пожар\
8 - 2 - (каско) стихийное бедствие\
9 - 3 - (дмс) амбулаторное лечение\
10 - 3 - (дмс) госпитализация\
11 - 3 - (дмс) диагностика\
12 - 3 - (дмс) стоматология\
13 - 4 - (страхование жизни) смерть\
14 - 4 - (страхование жизни) критическое заболевание\
15 - 4 - (страхование жизни) дожитие\
16 - 5 - (несчастный случай) травма\
17 - 5 - (несчастный случай) перелом\
18 - 5 - (несчастный случай) инвалидность\
19 - 6 - (страхование имущества) пожар\
20 - 6 - (страхование имущества) затопление\
21 - 6 - (страхование имущества) кража\
22 - 6 - (страхование имущества) повреждение имущества\
23 - 6 - (страхование имущества) стихийное бедствие\


In [841]:
claim_types_list = [
    (1, 1,"ДТП"),
    (2, 1, "Вред здоровью"),
    (3, 1, "Повреждение имущества"),
    (4, 2, "ДТП"),
    (5, 2, "Угон"),
    (6, 2, "Повреждение автомобиля"),
    (7, 2, "Пожар"),
    (8, 2, "Стихийное бедствие"),
    (9, 3, "Амбулаторное лечение"),
    (10, 3, "Госпитолизация"),
    (11, 3, "Диагностика"),
    (12, 3, "Стоматология"),
    (13, 4, "Смерть"),
    (14, 4, "Критическое заболевание"),
    (15, 4, "Дожитие"),
    (16, 5, "Травма"),
    (17, 5, "Перелом"),
    (18, 5, "Инвалидность"),
    (19, 6, "Пожар"),
    (20, 6, "Затопление"),
    (21, 6, "Кража"),
    (22, 6, "Повреждение имущества"),
    (23, 6, "Стихийное бедствие")
]

# как таблица
claim_types = pd.DataFrame(claim_types_list, columns=["claim_type", "policy_type_id", "type_name"])
claim_types.to_csv("claim_types.csv", index = False)

In [842]:
claim_statuses_list = [
    (1, "Выплачено"),
    (2, "В процессе"),
    (3, "Отклонено")
]

claim_statuses = pd.DataFrame(claim_statuses_list, columns=["status_id", "status"])
claim_statuses.to_csv("claim_statuses.csv", index = False)